**Exploratory notebook** used to figure out the merchant-name -> chain mapping and the card x month aggregation logic against the raw Visa parquet drop.

The reproducible version of this extraction now lives in [`pipeline/extract_card_month_csv.py`](../extract_card_month_csv.py) (DuckDB instead of dask -- full-file aggregation over the ~17GB parquet runs in seconds instead of minutes). Re-run that script, not this notebook, to regenerate `data/raw/card_month_4month.csv`.

In [15]:
import numpy as np
import pandas as pd
import dask.dataframe as dd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [16]:
df = dd.read_parquet("../../data/raw/datasprint_sample_data.parquet")
df.head()

,tran_id_raw,tran_id_gmt_tm,pymt_crd_acct_num_raw,prod_id_pltfrm_cd_vcis,transaction_type,transaction_pos_entry_mode,mrch_regn_cd,mrch_regn_nm,mrch_ctry_cd,mrch_ctry_nm,mrch_nm_raw,mrch_catg_cd,mrch_catg_nm,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,issr_jurn,issr_ctry_cd,issr_ctry_nm,crd_typ_cd,crd_typ_nm,channel_flg,cp_flag,prch_mnth_id,prch_dt,myweek,report_ctry,lau_enr,fua_enr,pstl_cd_enr
0,974297999,184525,8ff868023e4cc1ddfc3a389114fbedf71e2eb2bef877989,CN,POS,Tap to pay,03,EUROPE,616,POLAND,MAESTRIA 01,5812,EATING PLACES AND RESTAURANTS,TORUN,87-120,35.8800000000,Domestic,616,POLAND,C,CLASSIC,mobile,1,202506,2025-06-27,2025W26,616,TORUN,TORUN,87-100
1,974298000,151020,13ac806cdba2bcba5a191177c4ca8bb3039ad3daa691a1a,CN,POS,Tap to pay,03,EUROPE,616,POLAND,DINO BORUJA KOSCIELNA,5411,GROCERY STORES/SUPERMARKETS,BORUJA KOSCIE,64-300,68.9558400000,Domestic,616,POLAND,C,CLASSIC,cp_contactless,1,202506,2025-06-10,2025W24,616,NOWY TOMYSL,POZNAN,64-300
2,974298005,170503,0ec2fba8019d37fa25bb593c2e2c455e841bae9808b82b0,CN,POS,COF,03,EUROPE,616,POLAND,CDA,5815,DIGITAL GOODS BOOKSMOVIEMUSIC,Wroclaw,54-204,83.8690400000,Domestic,616,POLAND,C,CLASSIC,eci,0,202506,2025-06-20,2025W25,616,RADZANOWO,PLOCK,09-451
3,974298010,112502,4dce21c357c73738c8439e41bb2af683a706f92fef75a19,CN,POS,Tap to pay,03,EUROPE,616,POLAND,ORLEN STACJA NR 4547,5541,SERVICE STATIONS,KOSAKOWO,81-198,138.7948800000,Domestic,616,POLAND,C,CLASSIC,mobile,1,202506,2025-06-23,2025W26,616,WARSZAWA,WARSZAWA,00-539
4,974298019,071047,759c7db22a7f779fd86c553f523ee511588d62ae8b0f927,CN,POS,Tap to pay,03,EUROPE,616,POLAND,Sklep Wedkarski RED,5995,PET STORES/FOOD & SUPPLY,Wejherowo,84-200,167.8080000000,Domestic,616,POLAND,C,CLASSIC,mobile,1,202506,2025-06-18,2025W25,616,WEJHEROWO,GDANSK,84-200


In [17]:
df.columns

Index(['tran_id_raw', 'tran_id_gmt_tm', 'pymt_crd_acct_num_raw',
       'prod_id_pltfrm_cd_vcis', 'transaction_type',
       'transaction_pos_entry_mode', 'mrch_regn_cd', 'mrch_regn_nm',
       'mrch_ctry_cd', 'mrch_ctry_nm', 'mrch_nm_raw', 'mrch_catg_cd',
       'mrch_catg_nm', 'mrch_city_nm_raw', 'mrch_postal_code', 'cs_tran_amt',
       'issr_jurn', 'issr_ctry_cd', 'issr_ctry_nm', 'crd_typ_cd', 'crd_typ_nm',
       'channel_flg', 'cp_flag', 'prch_mnth_id', 'prch_dt', 'myweek',
       'report_ctry', 'lau_enr', 'fua_enr', 'pstl_cd_enr'],
      dtype='object')

In [60]:
sup_poland = df[(df['mrch_catg_cd'] == 5411) & (df['mrch_ctry_cd'] == 616) & (df['mrch_city_nm_raw'].str.lower() == 'poznan')]

In [61]:
df_sub = sup_poland[['pymt_crd_acct_num_raw', 'mrch_nm_raw', 'mrch_city_nm_raw', 'mrch_postal_code','cs_tran_amt', 'prch_dt', 'pstl_cd_enr', 'prch_mnth_id']]

In [62]:
df_sub.head()

,pymt_crd_acct_num_raw,mrch_nm_raw,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,prch_dt,pstl_cd_enr,prch_mnth_id
93,15a773c0860dfcf207182806330ed39bad7c4cacec09b59,LIDL MIESZKA I,POZNAN,61-005,47.6008000000,2025-07-24,61-693,202507
733,aefb184a5098b0a6eb6ad37dc2f37e44f7f4bc69d5b13d1,LIDL 1884,POZNAN,60-538,285.3288000000,2025-01-17,60-842,202501
990,5a12747457b4ce9d2c53f70018ff1dccdf2492f8fd9f420,CARREFOUR HIPERMARKET,POZNAN,60-201,41.3576800000,2025-10-17,62-080,202510
1246,441ca725a39a19537153433feeebff080c5f31bdb30b055,NETTO 5602 K.1,POZNAN,60-533,65.5408000000,2025-10-31,61-568,202510
1336,a1171b427e3b13a7e74d21a2e9b92f111e4e4ccfac2b0e6,LIDL NARAMOWICKA,POZNAN,61-005,112.7736000000,2025-10-13,64-500,202510


In [63]:
display(df_sub[(df_sub['mrch_nm_raw'].str.lower().str.contains('carrefour'))].head())

,pymt_crd_acct_num_raw,mrch_nm_raw,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,prch_dt,pstl_cd_enr,prch_mnth_id
990,5a12747457b4ce9d2c53f70018ff1dccdf2492f8fd9f420,CARREFOUR HIPERMARKET,POZNAN,60-201,41.3576800000,2025-10-17,62-080,202510
1399,c1e2c89aadfd0f3e60cf0e55eabdb4e3921ecda1fb22aa9,CARREFOUR PLP245,POZNAN,61-696,49.5622400000,2025-08-21,61-696,202508
2488,cd3b6907cc30029536239dcd95b64652f9c941e88057a4e,CARREFOUR HIPERMARKET,POZNAN,60-201,44.5077600000,2025-12-11,60-335,202512
4554,74f94a8a08bf98b9e06d24a7f61f9ceebba2a98a2b60b38,CARREFOUR HIPERMARKET,POZNAN,60-201,431.2334400000,2026-01-08,61-693,202601
5606,4ed4f34187dfe37565c9605efaa9f05bd868c91be5988c9,CARREFOUR HIPERMARKET,POZNAN,60-201,315.9261600000,2025-04-25,61-131,202504


In [64]:
display(df_sub[(df_sub['mrch_nm_raw'].str.lower().str.startswith('kaufland'))].head())
display(df_sub[(df_sub['mrch_nm_raw'].str.lower().str.startswith('lidl'))].head())
display(df_sub[(df_sub['mrch_nm_raw'].str.lower().str.startswith('jmp'))].head())
display(df_sub[(df_sub['mrch_nm_raw'].str.lower().str.startswith('netto'))].head())
display(df_sub[(df_sub['mrch_nm_raw'].str.lower().str.startswith('stokrotka'))].head())
display(df_sub[(df_sub['mrch_nm_raw'].str.lower().str.startswith('aldi'))].head())

,pymt_crd_acct_num_raw,mrch_nm_raw,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,prch_dt,pstl_cd_enr,prch_mnth_id
14060,d1cd538916d73fa20e28ab24400419dfb2f591e7615c161,KAUFLAND 01,POZNAN,60-335,46.3900800000,2025-04-22,61-451,202504
23654,d0ef153a74abfd6a9ccd1099641c2d13e1d949bf78db598,KAUFLAND 01,POZNAN,60-335,60.6997600000,2025-10-30,60-335,202510
31645,6f5a8480fe1c3752127406d7b71b6cb5ad4c50f8bde7c43,KAUFLAND 01,POZNAN,60-335,135.4700000000,2025-12-21,61-770,202512
31675,593ad32b36eced3f2f00a589bbafb0d4617b1cb22db26f4,KAUFLAND 01,POZNAN,60-335,324.8133600000,2025-12-14,61-614,202512
40643,f3e6e0134f9be6ffbeb64f9c47bbc728b2b91cb0359b462,KAUFLAND 01,POZNAN,60-335,285.7428000000,2025-07-17,93-180,202507


,pymt_crd_acct_num_raw,mrch_nm_raw,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,prch_dt,pstl_cd_enr,prch_mnth_id
93,15a773c0860dfcf207182806330ed39bad7c4cacec09b59,LIDL MIESZKA I,POZNAN,61-005,47.6008000000,2025-07-24,61-693,202507
733,aefb184a5098b0a6eb6ad37dc2f37e44f7f4bc69d5b13d1,LIDL 1884,POZNAN,60-538,285.3288000000,2025-01-17,60-842,202501
1336,a1171b427e3b13a7e74d21a2e9b92f111e4e4ccfac2b0e6,LIDL NARAMOWICKA,POZNAN,61-005,112.7736000000,2025-10-13,64-500,202510
1572,7289dd51dd4994926415741011ca0e64157759a3a0d820d,LIDL 2281,POZNAN,61-055,26.2163200000,2025-08-13,60-101,202508
2682,d40d102e1fa610f552dcd6f2f130cbecb3479b85f62f731,LIDL WARSZAWSKA,POZNAN,61-055,188.7472000000,2025-12-22,61-031,202512


,pymt_crd_acct_num_raw,mrch_nm_raw,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,prch_dt,pstl_cd_enr,prch_mnth_id
2190,0c8f24c206b8861eab7b6045794ec0055089f9df8f7ba3b,JMP S.A. BIEDRONKA 1011,POZNAN,60-341,2.5392000000,2026-05-20,<NA>,202605
2191,0c8f24c206b8861eab7b6045794ec0055089f9df8f7ba3b,JMP S.A. BIEDRONKA 1011,POZNAN,60-341,20.0155200000,2026-05-20,<NA>,202605
2727,5f7e4e2479b9ac55e43c6c451fffbc8cdb04c68b670f49c,JMP S.A. BIEDRONKA 1012,POZNAN,61-455,26.5953600000,2025-08-30,61-424,202508
3307,94d4430bccd9f5df11fbb21c7d2e9fda5839dd51ff3a936,JMP S.A. BIEDRONKA 3109,POZNAN,60-688,593.0264800000,2025-02-08,62-002,202502
3549,ebdbce94a56a9c7c646b68aa52780dfb78f3f4ab27c2b21,JMP S.A. BIEDRONKA 7565,POZNAN,60-335,40.0310400000,2026-04-13,85-792,202604


,pymt_crd_acct_num_raw,mrch_nm_raw,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,prch_dt,pstl_cd_enr,prch_mnth_id
1246,441ca725a39a19537153433feeebff080c5f31bdb30b055,NETTO 5602 K.1,POZNAN,60-533,65.5408000000,2025-10-31,61-568,202510
1418,c274d692ff1f7e1adc1dec31dcbdbc91651c73255018589,NETTO 4479 K.2,POZNAN,61-614,16.1368000000,2025-08-20,61-614,202508
1611,f656851f5a92c2e2aa96c389a34a476719fedaa773d9f09,NETTO 5270 K.1,POZNAN,60-335,18.7569600000,2025-08-21,61-433,202508
15668,e17178150ba73a47dcd7aa7101de40789032372ed74ed58,NETTO 5304 SCO K.2,POZNAN,60-853,54.5836000000,2025-10-18,61-005,202510
18098,22e06e5ac11c2086d20469b33b9bc3215d403be08dd535e,NETTO 5579 SCO K.4,POZNAN,60-101,70.6596800000,2025-08-19,60-539,202508


,pymt_crd_acct_num_raw,mrch_nm_raw,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,prch_dt,pstl_cd_enr,prch_mnth_id
11460,8e402058f67ff576bc9f9af49ff2830d48fb923c6a622a6,STOKROTKA NR 0122,POZNAN,60-335,17.9124000000,2026-04-02,64-320,202604
15980,eeac9403c3d48b5d46155c3080c52d852359e352063dce1,STOKROTKA NR 1274,POZNAN,61-815,47.0175200000,2025-02-10,61-538,202502
29080,9df18dde0b5b502a4f8bb0a7aaa117e64da9c2b67c3b65f,STOKROTKA NR 0291,POZNAN,60-335,2.6772000000,2025-04-24,60-335,202504
33086,74ca9d165ff7096dcc59c746c34cd8f53efc83d91fd4ec5,STOKROTKA NR 0292,POZNAN,61-815,41.4515200000,2025-02-27,61-442,202502
37621,972091b92d08544e4411e0fc2751efc1de84561816a92b9,STOKROTKA NR 0302,POZNAN,60-335,56.7456000000,2025-09-18,61-395,202509


,pymt_crd_acct_num_raw,mrch_nm_raw,mrch_city_nm_raw,mrch_postal_code,cs_tran_amt,prch_dt,pstl_cd_enr,prch_mnth_id
6344,ca68aca80e5291e1a1bb066100f84b672d75cc90b0c4f89,ALDI Sp. z o.o. 126,Poznan,61-887,20.2473600000,2025-10-07,61-887,202510
6357,649f302ef912f4a41af749a6556c4d089e8023163ab495e,ALDI SP. Z O.O. 04,POZNAN,61-248,289.8570400000,2025-10-14,60-301,202510
10545,3246519d3d46e917c1f3fb9f7e49dd56686d991c69c5032,ALDI SP. Z O.O. 04,POZNAN,61-248,25.7857600000,2026-05-16,<NA>,202605
15143,9add13f16fa163544be05d5f18a9ee013e64715c0a163ef,ALDI SP. Z O.O. 05,POZNAN,61-248,46.5317600000,2026-04-23,61-318,202604
23266,649f302ef912f4a41af749a6556c4d089e8023163ab495e,ALDI SP. Z O.O. 04,POZNAN,61-248,103.6932000000,2025-09-05,60-335,202509


In [ ]:
df_sub[df_sub['mrch_nm_raw'].str.lower().str.startswith('kaufland')]['mrch_nm_raw'].unique().compute()

In [ ]:
df_sub[df_sub['mrch_nm_raw'].str.lower().str.startswith('kaufland')]['mrch_postal_code'].unique().compute()

In [ ]:
sup_poland.head(5)

In [27]:
MERCHANT_PREFIXES = {
    'lidl': 'lidl',
    'kaufland': 'kaufland',
    'biedronka': 'jmp',
    'netto': 'netto',
    'stokrotka': 'stokrotka',
    'aldi': 'aldi',
    'carrefour': 'carrefour',
}

def build_card_month_aggregate(data, start_month=None, end_month=None):
    """
    Card x month aggregate: sum of spend per merchant + a 3-way lidl flag,
    pstl_cd_enr, crd_typ_nm, and avg transaction value.

    data: dask DataFrame with pymt_crd_acct_num_raw, mrch_nm_raw, cs_tran_amt,
          prch_mnth_id, pstl_cd_enr, crd_typ_nm
    start_month/end_month: inclusive YYYYMM bounds (int), e.g. 202501. None = unbounded.

    flag_lidl values:
      'lidl_only'      - only lidl transactions (among tracked merchants) that month
      'lidl_and_other' - lidl plus at least one other tracked merchant
      'other_only'     - at least one other tracked merchant, no lidl
      'none'           - no transaction at any tracked merchant that month
                         (e.g. only an untracked merchant)

    avg_txn_value = (spd_lidl + spd_kaufland + ... + spd_carrefour) / count of
                    transactions that matched one of the tracked merchants that
                    month. NaN when flag_lidl == 'none' (no tracked txns).
    """
    cols = ['pymt_crd_acct_num_raw', 'mrch_nm_raw', 'cs_tran_amt', 'prch_mnth_id',
            'pstl_cd_enr', 'crd_typ_nm']
    d = data[cols].copy()

    if start_month is not None:
        d = d[d['prch_mnth_id'] >= start_month]
    if end_month is not None:
        d = d[d['prch_mnth_id'] <= end_month]

    d['cs_tran_amt'] = d['cs_tran_amt'].astype('float64')
    mrch_lower = d['mrch_nm_raw'].str.lower()

    agg_dict = {'pstl_cd_enr': 'max', 'crd_typ_nm': 'max'}
    spd_cols = []
    lidl_mask = None
    other_mask = None
    for name, prefix in MERCHANT_PREFIXES.items():
        mask = mrch_lower.str.startswith(prefix)
        spd_col = f'spd_{name}'
        d[spd_col] = d['cs_tran_amt'].where(mask, 0)
        agg_dict[spd_col] = 'sum'
        spd_cols.append(spd_col)

        if name == 'lidl':
            lidl_mask = mask
            d['is_lidl'] = mask.astype('int8')
            agg_dict['is_lidl'] = 'max'
        else:
            other_mask = mask if other_mask is None else (other_mask | mask)

    d['is_other'] = other_mask.astype('int8')
    agg_dict['is_other'] = 'max'

    d['is_tracked'] = (lidl_mask | other_mask).astype('int8')
    agg_dict['is_tracked'] = 'sum'

    result = (
        d.groupby(['pymt_crd_acct_num_raw', 'prch_mnth_id'])
         .agg(agg_dict)
         .reset_index()
         .rename(columns={'pymt_crd_acct_num_raw': 'card', 'prch_mnth_id': 'month',
                           'is_tracked': 'tracked_txn_count'})
    )

    result['flag_lidl'] = 'none'
    result['flag_lidl'] = result['flag_lidl'].mask((result['is_lidl'] == 1) & (result['is_other'] == 0), 'lidl_only')
    result['flag_lidl'] = result['flag_lidl'].mask((result['is_lidl'] == 1) & (result['is_other'] == 1), 'lidl_and_other')
    result['flag_lidl'] = result['flag_lidl'].mask((result['is_lidl'] == 0) & (result['is_other'] == 1), 'other_only')

    tracked_spend = result[spd_cols].sum(axis=1)
    result['avg_txn_value'] = tracked_spend / result['tracked_txn_count']

    result = result.drop(columns=['is_lidl', 'is_other', 'tracked_txn_count'])

    return result

In [ ]:
sup_poland['prch_mnth_id'].unique().compute()

In [28]:
sample = sup_poland   # first 5 row-group partitions only, still lazy
a = build_card_month_aggregate(sample, start_month=202501, end_month=202504).compute()

In [29]:
a.shape

(77021, 13)

In [31]:
a.to_csv('test_sample.csv')

In [30]:
a[a['flag_lidl'] == 'lidl_and_other'].head(5)

,card,month,pstl_cd_enr,crd_typ_nm,spd_lidl,spd_kaufland,spd_biedronka,spd_netto,spd_stokrotka,spd_aldi,spd_carrefour,flag_lidl,avg_txn_value
0,8ab0370d43a350bb558b18a0d79d0ab9a9d9edb7aea976f,202503,60-533,CLASSIC,1231.15136,0.0,28.45744,336.904,0.00000,0.00000,0.0,lidl_and_other,228.073257
1,c071014c63c694fe20eb2ab510751410094735a3f327002,202501,62-090,CLASSIC,5.13912,0.0,12.48072,0.000,0.00000,0.00000,0.0,lidl_and_other,8.809920
2,8b0540d1b557ad6e70207ba5bbedf0516cf74255cf58421,202503,61-136,CLASSIC,175.28576,0.0,941.11952,0.000,86.42480,0.00000,0.0,lidl_and_other,75.176880
5,21d328495ae0694fcd47f538ccc8c95bd7060c2d71ec42b,202501,61-655,CLASSIC,956.70432,0.0,23.16744,0.000,0.00000,0.00000,0.0,lidl_and_other,97.987176
6,179c4e150b6755c0923e6903a337338d1f9d0df0ba7684f,202501,61-541,CLASSIC,754.98696,0.0,324.69008,0.000,31.89456,59.36208,0.0,lidl_and_other,117.093368


In [59]:
for code in a['pstl_cd_enr'].str.split('-'):
    if not isinstance(code, (list, np.ndarray)):
        
        continue
    if len(code[0]) != 2 or len(code[1]) != 3:
        print(code)

['45130']
['10551']
['61754']
['61754']
['E91']
['02797']
['K67']
['N15 4QD']
['61754']
['RH2 9AE']
['63000']
['12345']
['62081']
['61754']
['3811']
['9050']
['35214']
['61754']
['LV', '1013']
['040 01']
['59510']
['62081']
['10551']
['61754']
['14129']
['1062']
['61754']
['24148']
['2332']
['04177']
['0164']
['01156']
['W8 5SF']
['92600']
['61754']
['6903']
['390 02']
['7130']
['33613']
['12524']
['075100']
['SK13 8HB']
['10249']
['1000', '117']
['8081']
['61754']
['00026']
['370 01']
['040 01']
['61754']
['28002']
['G51 4BP']
['500 11']
['T12']
['LE13 1RB']
['DA10 0BW']
['AL10 9BY']
['3511']
['FY8 5RW']
['61754']
['05090']
['NN16 8JY']
['25840']
['13581']
['LT', '91177']
['KT91NU']
['00076']
['62081']
['BH14 9DQ']
['28055']
['RM17 5XZ']
['61754']
['42389']
['61754']
['63069']
['17034']
['TW5 9SL']
['221']
['51107']
['783 91']
['W10 6HJ']
['W9 3NA']
['62081']
['L', '7543']
['186 00']
['7190']
['143 00']
['61754']
['61754']
['RG21 7DA']
['LT', '07160']
['EN2 6HU']
['01127']
['40880']
[

In [ ]:
a['card'].nunique()

In [ ]:
# pick your period here (inclusive, YYYYMM), then compute
card_month_agg = build_card_month_aggregate(sup_poland, start_month=202501, end_month=202512).compute()
card_month_agg.head(10)